# Filter Evals   
---   
This code looks in the `model_outputs` folder and for each folder, it runs through the various files and creates two outputs:
1) Correct Solutions
2) Incorrect Solutions

In [1]:
import json
from pathlib import Path
from utils.parsing import extract_solution, coerce_response
from utils.exceptions import IllegalMoveException

In [2]:
def evaluate_file(fp: Path):
    task = fp.stem.split('_')[0]
    res = {"Correct": 0, "Incorrect": 0, "Thrown Away": 0}
    correct, incorrect = [], []
    data = json.load(fp.open())
    for e in data:
        try:
            prompt = e["prompt"]
            resp   = e["model_response"]
            gt     = e["info"]["answer"]
            if task in ("bestmove", "worstmove"):
                answer     = gt["answer"]
                candidates = gt["candidates"]
                pred = coerce_response(extract_solution(resp), "choose_from_n")
                if pred == answer:
                    res["Correct"] += 1; correct.append(e)
                elif pred in candidates:
                    res["Incorrect"] += 1; incorrect.append(e)
                else:
                    raise IllegalMoveException()
            elif task == "legalmoves":
                pred = coerce_response(extract_solution(resp), "produce_list")
                if set(pred) == set(gt) and len(pred) == len(gt):
                    res["Correct"] += 1; correct.append(e)
                else:
                    res["Incorrect"] += 1; incorrect.append(e)
            elif task == "predictmove":
                pred = coerce_response(extract_solution(resp), "predict_singlemove")
                if pred in gt:
                    sorted_gt = sorted(gt.items(), key=lambda x: x[1])
                    idx = next(i for i,(m,_) in enumerate(sorted_gt) if m == pred)
                    rank = idx / len(sorted_gt)
                    if rank > 0.7:
                        res["Correct"] += 1; correct.append(e)
                    else:
                        res["Incorrect"] += 1; incorrect.append(e)
                else:
                    raise IllegalMoveException()
            else:
                # unknown task; skip
                continue
        except Exception:
            res["Thrown Away"] += 1
    return task, res, correct, incorrect

base = Path("model_outputs")
for folder in base.iterdir():
    if not folder.is_dir(): continue
    agg = {"Correct": 0, "Incorrect": 0, "Thrown Away": 0}
    for fp in folder.glob("*.json"):
        task, res, corr, incorr = evaluate_file(fp)
        agg["Correct"]     += res["Correct"]
        agg["Incorrect"]   += res["Incorrect"]
        agg["Thrown Away"] += res["Thrown Away"]
        # write out per-file JSONL
        out_corr = folder / f"{fp.stem}_correct.jsonl"
        out_inc  = folder / f"{fp.stem}_incorrect.jsonl"
        with out_corr.open("w") as f:
            for e in corr: f.write(json.dumps(e) + "\n")
        with out_inc.open("w") as f:
            for e in incorr: f.write(json.dumps(e) + "\n")
    print(f"{folder.name}: {agg['Correct']} correct, {agg['Incorrect']} incorrect, {agg['Thrown Away']} thrown away")

llmchess-llama31-8b-400: 203 correct, 878 incorrect, 519 thrown away
llmchess-llama31-8b-sft-mmxl-400: 440 correct, 893 incorrect, 267 thrown away
llmchess-llama4-maverick-400: 340 correct, 925 incorrect, 335 thrown away
llmchess-qwen25-7b-400: 175 correct, 861 incorrect, 564 thrown away
llmchess-qwen25-7b-grpo-datamix-2-400: 563 correct, 886 incorrect, 151 thrown away
llmchess-qwen25-7b-grpo-mmxl-400: 410 correct, 1018 incorrect, 172 thrown away
llmchess-qwen25-7b-sft-dm2-v2-400: 373 correct, 865 incorrect, 362 thrown away
llmchess-qwen25-7b-sft-mmxl-400: 430 correct, 844 incorrect, 326 thrown away
